# 08. 열회피 **보행** 경로 분석 — 과제 2·3 (성남)

★수정: 03 도로망은 **차량망**이라 보행 대회에 부적합 → **OSM 보행망**(footway·보행자도로·공원길, 자동차전용 제외)으로 교체.

- 링크 열노출 = exposure_rate(UTCI) × 일사노출(SVF·NDVI 그늘) × 체류시간. 발한임계(32°C) 지수 급증.
- 곤란구간 = 보행 매개중심성 × 열노출 (피할 수 없는 뜨거운 보행 길목).

산출물: `data/results/bottleneck_walk.geojson`, `output/figures/fig11_routes.png`

In [1]:
# ── 실행 환경 가드 (하이브리드 전략, 실행순서.md §1 참조) ──────────────
# 이 노트북은 '로컬 사전계산' 단계입니다. COMPAS에는 ['osmnx'] 이(가) 없으므로
# 그 경우 아래 셀들은 자동으로 건너뛰고(LOCAL_RUN=False) 동봉 산출물만 검증합니다.
# 산출물은 제출 데이터 ZIP에 포함되어 있어 후속 노트북은 그대로 실행됩니다.
import importlib.util, pathlib
_DEPS = ['osmnx']
_missing = [m for m in _DEPS if importlib.util.find_spec(m) is None]
LOCAL_RUN = not _missing
_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
if LOCAL_RUN:
    print("[로컬 모드] 의존성 충족 - 전체 셀을 실행합니다.")
else:
    print("[COMPAS 모드] 미설치 의존성:", _missing, "-> 아래 셀은 건너뜁니다.")
    print("이 단계의 동봉 산출물 검증:")
    _o = _ROOT / "data/results/bottleneck_walk.geojson"
    _r = _ROOT / "data/local/restore_bottleneck_walk.geojson.gz"
    if _o.exists():
        print("  [OK] data/results/bottleneck_walk.geojson (%.1f MB)" % (_o.stat().st_size / 1e6))
    elif _r.exists():
        import gzip as _gz, shutil as _sh
        _o.parent.mkdir(parents=True, exist_ok=True)
        with _gz.open(_r, "rt", encoding="utf-8") as _f, open(_o, "w", encoding="utf-8") as _w:
            _sh.copyfileobj(_f, _w)
        print("  [복원] restore_bottleneck_walk.geojson.gz -> bottleneck_walk.geojson")
    else:
        print("  [!!] bottleneck_walk.geojson 없음 - 후속 14번은 건너뜁니다")


[로컬 모드] 의존성 충족 - 전체 셀을 실행합니다.


In [2]:
if LOCAL_RUN:
    import sys, pathlib
    sys.path.append(str(pathlib.Path.cwd().parent))
    import numpy as np, pandas as pd, geopandas as gpd, networkx as nx, osmnx as ox
    from shapely.geometry import Point, LineString
    from src.config import DATA_PROCESSED, DATA_RESULTS, OUT_FIGURES, CRS_LANDSAT
    from src.network.routing import nearest_node, route_metrics, exposure_rate, WALK_SPEED

    # ④ 도로폭(OSM highway 등급 = 인도 땡볕 노출) 계수
    HW = {"motorway": 1.3, "trunk": 1.3, "primary": 1.25, "secondary": 1.15, "tertiary": 1.1,
          "residential": 1.0, "living_street": 0.9, "unclassified": 1.0, "service": 0.95,
          "pedestrian": 0.85, "footway": 0.8, "path": 0.8, "steps": 0.8, "track": 0.85, "cycleway": 0.9}
    def hwf(h):
        if isinstance(h, list):
            h = h[0] if h else None
        return HW.get(h, 1.0)

    Gw = ox.graph_from_bbox(bbox=(127.04, 37.315, 127.164, 37.431), network_type="walk")
    node_xy = {n: (d["x"], d["y"]) for n, d in Gw.nodes(data=True)}

    # 격자 열속성 (+ water_frac 하천 냉섬)
    g = gpd.read_file(DATA_RESULTS / "heat_index_utci.geojson")[["grid_id", "utci_daymean", "svf", "geometry"]]
    g = g.merge(gpd.read_file(DATA_PROCESSED / "grid_features.geojson")[["grid_id", "ndvi", "water_frac"]], on="grid_id").to_crs(CRS_LANDSAT)

    erecs = [(u, v, dt.get("length", 0.0), dt.get("highway")) for u, v, dt in Gw.edges(data=True) if u != v]
    mp = [Point((node_xy[u][0]+node_xy[v][0])/2, (node_xy[u][1]+node_xy[v][1])/2) for u, v, _, _ in erecs]
    mids = gpd.GeoDataFrame({"eid": range(len(erecs))}, geometry=mp, crs=4326).to_crs(CRS_LANDSAT)
    lj = (gpd.sjoin_nearest(mids, g, how="left")
          .sort_values(["eid", "index_right"]).drop_duplicates("eid").reset_index(drop=True))   # 동률 고정

    G = nx.Graph()
    for i, (u, v, length, hw) in enumerate(erecs):
        ut, svf, nd, wf = lj.utci_daymean[i], lj.svf[i], lj.ndvi[i], lj.water_frac[i]
        # 일사노출 = 그늘(SVF·가로수)·하천 냉섬 낮춤 × 도로폭(등급) 높임
        shade = np.clip(svf - 0.5 * np.clip(nd, 0, 1) - 0.5 * np.clip(wf, 0, 1), 0.2, 1.0)
        sun = float(np.clip(shade * hwf(hw), 0.15, 1.4))
        t = length / WALK_SPEED
        dose = float(exposure_rate(ut)) * sun * t if np.isfinite(ut) else t
        if not G.has_edge(u, v) or G[u][v]["length"] > length:
            G.add_edge(u, v, length=length, time=t, utci=ut, sun=sun, dose=dose)
    main = max(nx.connected_components(G), key=len)
    G = G.subgraph(main).copy()
    node_xy = {n: node_xy[n] for n in G.nodes}
    print("보행망:", G.number_of_nodes(), "노드", G.number_of_edges(),
          "| 일사노출(도로폭·하천 반영)", round(min(G[a][b]['sun'] for a, b in G.edges), 2),
          "~", round(max(G[a][b]['sun'] for a, b in G.edges), 2))

보행망: 17940 노드 23295 | 일사노출(도로폭·하천 반영) 0.16 ~ 1.19


In [3]:
if LOCAL_RUN:
    LM = {"판교역": (127.1116, 37.3948), "정자역": (127.1085, 37.3665),
          "서현역": (127.1230, 37.3849), "미금역": (127.1085, 37.3502)}
    ODs = [("판교역", "정자역"), ("서현역", "미금역")]
    results = {}
    for o, d in ODs:
        no, nd = nearest_node(node_xy, *LM[o]), nearest_node(node_xy, *LM[d])
        ps, pc = nx.shortest_path(G, no, nd, weight="length"), nx.shortest_path(G, no, nd, weight="dose")
        ms, mc = route_metrics(G, ps), route_metrics(G, pc)
        results[(o, d)] = (ps, pc, ms, mc)
        print(f"[{o}→{d}] 최단 {ms['length_m']:.0f}m·UTCI{ms['utci_mean']:.1f}·dose{ms['dose']:.0f} | "
              f"열회피 {mc['length_m']:.0f}m·dose{mc['dose']:.0f} → 우회+{100*(mc['length_m']-ms['length_m'])/ms['length_m']:.0f}% / 열노출 {100*(ms['dose']-mc['dose'])/ms['dose']:.0f}%↓")

[판교역→정자역] 최단 3596m·UTCI35.4·dose6407 | 열회피 3849m·dose4521 → 우회+7% / 열노출 29%↓
[서현역→미금역] 최단 4445m·UTCI35.3·dose7484 | 열회피 4637m·dose5407 → 우회+4% / 열노출 28%↓


In [4]:
if LOCAL_RUN:
    bc = nx.edge_betweenness_centrality(G, k=300, weight="dose", seed=42)
    rows = []
    for (a, b), c in bc.items():
        e = G[a][b]
        if np.isfinite(e["utci"]):
            geom = LineString([node_xy[a], node_xy[b]])
            rows.append({"betweenness": c, "utci": e["utci"], "score": c * float(exposure_rate(e["utci"])), "geometry": geom})
    bdf = gpd.GeoDataFrame(rows, crs=4326)
    bdf["bottleneck"] = bdf["score"] / bdf["score"].max()
    print("곤란구간 상위 UTCI/betweenness:")
    print(bdf.nlargest(5, "bottleneck")[["betweenness", "utci", "bottleneck"]].round(3).to_string(index=False))

곤란구간 상위 UTCI/betweenness:
 betweenness  utci  bottleneck
       0.003 35.52       1.000
       0.003 35.51       0.996
       0.003 35.51       0.996
       0.002 35.68       0.948
       0.002 35.67       0.943


In [5]:
if LOCAL_RUN:
    DATA_RESULTS.mkdir(parents=True, exist_ok=True)
    bdf[["betweenness", "utci", "bottleneck", "geometry"]].to_file(DATA_RESULTS / "bottleneck_walk.geojson", driver="GeoJSON")

    import matplotlib.pyplot as plt
    from src.viz.fonts import setup_font
    setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
    edges_all = gpd.GeoSeries([LineString([node_xy[a], node_xy[b]]) for a, b in G.edges], crs=4326)
    fig, ax = plt.subplots(1, 2, figsize=(17, 9))
    edges_all.plot(ax=ax[0], color="#dddddd", linewidth=0.4)
    def pline(path):
        return gpd.GeoSeries([LineString([node_xy[n] for n in path])], crs=4326)
    ps, pc, ms, mc = results[("판교역", "정자역")]
    pline(ps).plot(ax=ax[0], color="#1f77b4", linewidth=3, label=f"최단 {ms['length_m']:.0f}m")
    pline(pc).plot(ax=ax[0], color="#2ca02c", linewidth=3, label=f"열회피 {mc['length_m']:.0f}m")
    ax[0].legend(); ax[0].set_title("판교역→정자역 보행: 최단(파랑) vs 열회피(초록)"); ax[0].set_axis_off()
    edges_all.plot(ax=ax[1], color="#eeeeee", linewidth=0.4)
    bdf.sort_values("bottleneck").plot(ax=ax[1], column="bottleneck", cmap="YlOrRd", linewidth=1.2, legend=True, legend_kwds={"shrink": 0.5})
    ax[1].set_title("보행 곤란구간 (피할 수 없는 뜨거운 길목)"); ax[1].set_axis_off()
    plt.tight_layout(); plt.savefig(OUT_FIGURES / "fig11_routes.png", dpi=120); plt.close()